In [1]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
from scipy.stats import qmc

### Function Description

You’re optimising a cake recipe using a black-box function with five ingredient inputs, for example flour, sugar, eggs, butter and milk. Each recipe is evaluated with a combined score based on flavour, consistency, calories, waste and cost, where each factor contributes negative points as judged by an expert taster. This means the total score is negative by design. 

To frame this as a maximisation problem, your goal is to bring that score as close to zero as possible or, equivalently, to maximise the negative of the total sum.

In [2]:
# Load data
X = np.load('initial_inputs.npy')
Y = np.load('initial_outputs.npy').ravel()

In [3]:
df = pd.DataFrame(X, columns=['x1', 'x2', 'x3', 'x4', 'x5'])
df['target'] = Y

print(df.describe())

              x1         x2         x3         x4         x5     target
count  20.000000  20.000000  20.000000  20.000000  20.000000  20.000000
mean    0.547765   0.562570   0.467217   0.534433   0.423745  -1.495390
std     0.303015   0.283617   0.315292   0.280259   0.297199   0.460664
min     0.021735   0.114404   0.016523   0.045613   0.004911  -2.571170
25%     0.254775   0.304453   0.165252   0.329066   0.111981  -1.713639
50%     0.624060   0.617630   0.435052   0.621691   0.506517  -1.446370
75%     0.773685   0.813638   0.714228   0.710318   0.628985  -1.227829
max     0.957740   0.931871   0.978806   0.961656   0.892819  -0.714265


In [4]:
# Kernel Setup and Model Fitting
initial_length_scales = np.array([1.0, 1.0, 1.0, 1.0, 1.0])
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 10.0), nu=2.5) + \
         WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-4, 1e1))

model = GaussianProcessRegressor(
    kernel=kernel, 
    normalize_y=True, 
    n_restarts_optimizer=30, 
    random_state=42
)

model.fit(X, Y)

# 5D Monte Carlo Random Search Space
np.random.seed(42)
grid_size = 100000 
search_grid = np.random.uniform(0, 1, size=(grid_size, 5))

mean, std = model.predict(search_grid, return_std=True)

# Acquisition Function (UCB & Beta Sweep)
betas = [0.1, 0.5, 1.0, 1.5, 2.0, 3.0]
sweep_results = []

for b in betas:
    ucb_scores = mean + (b * std)
    best_idx = np.argmax(ucb_scores)
    bp = search_grid[best_idx]
    
    sweep_results.append({
        'Beta': b,
        'x1': bp[0], 'x2': bp[1], 'x3': bp[2], 'x4': bp[3], 'x5': bp[4],
        'Pred Mean': mean[best_idx],
        'Uncertainty': std[best_idx]
    })

# Output Table and Model Diagnostics
df_sweep = pd.DataFrame(sweep_results)
print("--- FUNCTION 6: 5D CAKE RECIPE OPTIMIZATION ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)

--- FUNCTION 6: 5D CAKE RECIPE OPTIMIZATION ---
  Beta     x1     x2     x3     x4     x5  Pred Mean  Uncertainty
0.1000 0.5382 0.1960 0.5838 0.8231 0.0607    -0.5673       0.2491
0.5000 0.5382 0.1960 0.5838 0.8231 0.0607    -0.5673       0.2491
1.0000 0.5382 0.1960 0.5838 0.8231 0.0607    -0.5673       0.2491
1.5000 0.4766 0.2087 0.5408 0.9259 0.0271    -0.6283       0.3012
2.0000 0.4766 0.2087 0.5408 0.9259 0.0271    -0.6283       0.3012
3.0000 0.4429 0.1522 0.6550 0.9488 0.0233    -0.6854       0.3275

--- Optimized Kernel Parameters ---
Matern(length_scale=[0.367, 0.844, 0.797, 0.712, 0.673], nu=2.5) + WhiteKernel(noise_level=0.0385)


### Week 1 submission: x1 = 0.4766, x2 = 0.2087, x3 = 0.5408, x4 = 0.9259, x5 = 0.0271

* The Problem Framing: We are tuning a 5-ingredient cake recipe. The target is a combined penalty score from an expert taster (all negative numbers).
    * The Environment: A complex 5D space blending physical chemistry with human subjectivity.

* The Surrogate Model: Matern 2.5 + ARD + White Kernel.
    * Matern 2.5: Recipes are mostly smooth but can feature sharp drop-offs (e.g., adding too much flour ruins the cake suddenly, not linearly).
    * ARD (Automatic Relevance Determination): Crucial for 5D to determine which specific ingredients the expert taster is most sensitive to.
    * White Kernel: Absorbs the inherent "human noise" of a subjective taste test, preventing the model from overfitting to an inconsistent palate.

* Key Insights from the Model: The ARD length scale for x1​ was the smallest by far (0.367). The model mathematically identified that the taster is highly volatile to changes in Ingredient 1.
    * The "Salt" Validation: The model agreed with early visual data analysis, pushing x5​ down to near-zero (0.027) to maximize the score.
    * Realistic Confidence: The noise level settled at 0.0385, proving the White Kernel successfully caught the human subjectivity in the dataset.

* The Chosen Query (Beta = 2.0): [0.4766, 0.2087, 0.5408, 0.9259, 0.0271]
    * The Logic: An exploratory "Scout" move. Rather than playing it entirely safe at Beta 0.1, this query tests the limits of the recipe by pushing x4​ very high (0.92), crushing x5​ to its minimum (0.02), and making a bold reduction to the hyper-sensitive x1​ (0.47).

In [5]:
# ---------------------------------------------------
# WEEK1: NEW DATA HERE
# ---------------------------------------------------
w1_new_input = [0.4766, 0.2087, 0.5408, 0.9259, 0.0271]
w1_new_output = -0.5851186708242683

new_X = np.array(w1_new_input).reshape(1, -1)
new_Y = np.atleast_1d(w1_new_output)

X_updated_w1 = np.vstack((X, new_X))
Y_updated_w1 = np.append(Y, new_Y)

df = pd.DataFrame(X_updated_w1, columns=['x1', 'x2', 'x3', 'x4', 'x5'])
df['target'] = Y_updated_w1

print(df.describe())

              x1         x2         x3         x4         x5     target
count  21.000000  21.000000  21.000000  21.000000  21.000000  21.000000
mean    0.544377   0.545719   0.470721   0.553074   0.404857  -1.452044
std     0.295750   0.287019   0.307727   0.286208   0.302328   0.490976
min     0.021735   0.114404   0.016523   0.045613   0.004911  -2.571170
25%     0.258906   0.291470   0.187288   0.356552   0.110624  -1.702558
50%     0.618812   0.536336   0.443284   0.648324   0.501953  -1.356682
75%     0.770620   0.803484   0.708120   0.726272   0.614962  -1.209955
max     0.957740   0.931871   0.978806   0.961656   0.892819  -0.585119


In [6]:
initial_length_scales = np.array([1.0, 1.0, 1.0, 1.0, 1.0])
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 10.0), nu=2.5) + \
         WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-4, 1e1))

model = GaussianProcessRegressor(
    kernel=kernel, 
    normalize_y=True, 
    n_restarts_optimizer=30, 
    random_state=42
)

model.fit(X_updated_w1, Y_updated_w1)

np.random.seed(42)
grid_size = 100000 
search_grid = np.random.uniform(0, 1, size=(grid_size, 5))

mean, std = model.predict(search_grid, return_std=True)

betas = [0.1, 0.5, 1.0, 1.5, 2.0, 3.0]
sweep_results = []

for b in betas:
    ucb_scores = mean + (b * std)
    best_idx = np.argmax(ucb_scores)
    bp = search_grid[best_idx]
    
    sweep_results.append({
        'Beta': b,
        'x1': bp[0], 'x2': bp[1], 'x3': bp[2], 'x4': bp[3], 'x5': bp[4],
        'Pred Mean': mean[best_idx],
        'Uncertainty': std[best_idx]
    })

# Output Table and Model Diagnostics
df_sweep = pd.DataFrame(sweep_results)
print("--- FUNCTION 6: 5D CAKE RECIPE OPTIMIZATION ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)

--- FUNCTION 6: 5D CAKE RECIPE OPTIMIZATION ---
    Beta       x1       x2       x3       x4       x5  Pred Mean  Uncertainty
0.100000 0.538168 0.196035 0.583813 0.823088 0.060732  -0.540972     0.131905
0.500000 0.538168 0.196035 0.583813 0.823088 0.060732  -0.540972     0.131905
1.000000 0.528866 0.184657 0.726787 0.793502 0.127526  -0.569577     0.165631
1.500000 0.483705 0.064296 0.878958 0.828828 0.223290  -0.681718     0.244480
2.000000 0.473730 0.061961 0.907733 0.961873 0.298322  -0.746566     0.279797
3.000000 0.457182 0.005676 0.935365 0.974098 0.432408  -0.842906     0.322149

--- Optimized Kernel Parameters ---
Matern(length_scale=[0.412, 0.907, 0.877, 0.811, 0.785], nu=2.5) + WhiteKernel(noise_level=0.0335)


### Week 2 submission:

* Current Progress: Successfully improved the recipe score from -0.71 to -0.58. This confirms the model has identified a high-performing region, but the absolute ceiling of the "expert taster's" palate remains unknown.

* Maintaining β=2 to prioritize broad frontier searching over premature localization. 

* Next Query Coordinate (β=2.0): [0.47373024, 0.06196092, 0.90773256, 0.96187308, 0.29832166]

In [7]:
# ---------------------------------------------------
# WEEK2: NEW DATA HERE
# ---------------------------------------------------

w2_new_input = [0.473730, 0.061961, 0.907733, 0.961873, 0.298322]
w2_new_output = np.float64(-0.9135344071932381)

w2_new_X = np.atleast_2d(w2_new_input)
w2_new_Y = np.atleast_1d(w2_new_output)

X_updated_w2 = np.vstack((X_updated_w1, w2_new_X))
Y_updated_w2 = np.append(Y_updated_w1, w2_new_Y)

scaler_x = StandardScaler()
scaler_y = StandardScaler()

X_updated_w2_scaled = scaler_x.fit_transform(X_updated_w2)
Y_updated_w2_scaled = scaler_y.fit_transform(Y_updated_w2.reshape(-1, 1))

df = pd.DataFrame(X_updated_w2, columns=['x1', 'x2', 'x3', 'x4', 'x5'])
df['target'] = Y_updated_w2

# print(df.describe())
print(df.head(22))

          x1        x2        x3        x4        x5    target
0   0.728186  0.154693  0.732552  0.693997  0.056401 -0.714265
1   0.242384  0.844100  0.577809  0.679021  0.501953 -1.209955
2   0.729523  0.748106  0.679775  0.356552  0.671054 -1.672200
3   0.770620  0.114404  0.046780  0.648324  0.273549 -1.536058
4   0.618812  0.331802  0.187288  0.756238  0.328835 -0.829237
5   0.784958  0.910682  0.708120  0.959225  0.004911 -1.247049
6   0.145111  0.896685  0.896322  0.726272  0.236272 -1.233786
7   0.945069  0.288459  0.978806  0.961656  0.598016 -1.694343
8   0.125720  0.862725  0.028544  0.246605  0.751206 -2.571170
9   0.757594  0.355831  0.016523  0.434207  0.112433 -1.309116
10  0.536797  0.308781  0.411879  0.388225  0.522528 -1.144785
11  0.957740  0.235669  0.099146  0.156806  0.071317 -1.912677
12  0.629308  0.803484  0.811408  0.045613  0.110624 -1.622839
13  0.021735  0.428084  0.835939  0.489489  0.511082 -1.356682
14  0.439344  0.698924  0.426820  0.109476  0.877888 -2

In [8]:
initial_length_scales = np.array([1.0, 1.0, 1.0, 1.0, 1.0])
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 10.0), nu=2.5) + \
         WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-4, 1e1))

model = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=30, 
    random_state=42
)

model.fit(X_updated_w2_scaled, Y_updated_w2_scaled)

def acquisition_function(x_trial, beta, model):
    # L-BFGS-B passes the trial point in the scaled space
    x_trial = x_trial.reshape(1, -1)
    mean, std = model.predict(x_trial, return_std=True)
    ucb = mean[0] + (beta * std[0])
    return -ucb  # Minimize negative UCB to maximize UCB

betas = [0.1, 0.5, 1.0, 1.5, 2.0, 3.0]
sweep_results = []

sampler = qmc.Sobol(d=5, scramble=True, seed=42)
sample_raw = sampler.random(n=8192)
sample_scaled = scaler_x.transform(sample_raw)

for b in betas:
    # Find the best starting point 
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    # Precision Climbing with L-BFGS-B
    bounds = [scaler_x.transform([[0,0,0,0,0]])[0], scaler_x.transform([[1,1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    # Translate predictions back to raw physical units
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)
    
    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'x5': best_x_raw[4],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })

df_sweep = pd.DataFrame(sweep_results)
print("--- FUNCTION 6: 5D CAKE RECIPE OPTIMIZATION ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)


--- FUNCTION 6: 5D CAKE RECIPE OPTIMIZATION ---
    Beta       x1       x2       x3       x4       x5  Pred Mean  Uncertainty
0.100000 0.550537 0.300173 0.504927 0.799719 0.006655  -0.525729     0.134151
0.500000 0.549793 0.315541 0.501598 0.777361 0.000000  -0.527734     0.141012
1.000000 0.543514 0.337916 0.496825 0.744821 0.000000  -0.535328     0.150998
1.500000 0.532949 0.360868 0.492692 0.711446 0.000000  -0.550000     0.162705
2.000000 0.466917 0.519237 0.203142 1.000000 0.000000  -0.709927     0.239684
3.000000 0.403368 0.525421 0.000000 1.000000 0.000000  -0.883740     0.310978

--- Optimized Kernel Parameters ---
Matern(length_scale=[1.48, 2.97, 2.71, 2.81, 2.82], nu=2.5) + WhiteKernel(noise_level=0.026)


### Week 3 submission:

* Current Progress: Didn't improve the recipe compared to last time - the output was significantly worse. 

* Changing β=1 to start trying to opmtimize around the maximum. 

* Next Query Coordinate (β=1.0): [0.543514 , 0.337916 , 0.496826 , 0.744821 , 0]

In [9]:
# ---------------------------------------------------
# WEEK3: NEW DATA HERE
# ---------------------------------------------------

w3_new_input = [0.543514, 0.337916, 0.496825, 0.744821, 0.000000]
w3_new_output = -0.369336354412859

w3_new_X = np.atleast_2d(w3_new_input)
w3_new_Y = np.atleast_1d(w3_new_output)

X_updated_w3 = np.vstack((X_updated_w2, w3_new_X))
Y_updated_w3 = np.append(Y_updated_w2, w3_new_Y)

X_updated_w3_scaled = scaler_x.fit_transform(X_updated_w3)
Y_updated_w3_scaled = scaler_y.fit_transform(Y_updated_w3.reshape(-1, 1))

df = pd.DataFrame(X_updated_w3, columns=['x1', 'x2', 'x3', 'x4', 'x5'])
df['target'] = Y_updated_w3

print(df.describe())

              x1         x2         x3         x4         x5     target
count  23.000000  23.000000  23.000000  23.000000  23.000000  23.000000
mean    0.541268   0.515651   0.490856   0.579185   0.382623  -1.381556
std     0.282371   0.294186   0.307206   0.288137   0.300903   0.529542
min     0.021735   0.061961   0.016523   0.045613   0.000000  -2.571170
25%     0.345536   0.289965   0.264553   0.372389   0.090971  -1.698451
50%     0.543514   0.428084   0.496825   0.679021   0.328835  -1.309116
75%     0.764107   0.798581   0.720336   0.750530   0.606489  -1.040271
max     0.957740   0.931871   0.978806   0.961873   0.892819  -0.369336


In [10]:
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 10.0), nu=2.5) + \
         WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-10, 1e1))

model = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=30, 
    random_state=42
)

model.fit(X_updated_w3_scaled, Y_updated_w3_scaled)

betas = [0.10 , 0.25, 0.50, 0.75, 1.0, 1.5]
sweep_results = []

for b in betas:
    # Find the best starting point 
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    # Precision Climbing with L-BFGS-B
    bounds = [scaler_x.transform([[0,0,0,0,0]])[0], scaler_x.transform([[1,1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    # Translate predictions back to raw physical units
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)
    
    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'x5': best_x_raw[4],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })

df_sweep = pd.DataFrame(sweep_results)
print("--- FUNCTION 6: 5D CAKE RECIPE OPTIMIZATION ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)

--- FUNCTION 6: 5D CAKE RECIPE OPTIMIZATION ---
    Beta       x1       x2       x3       x4       x5  Pred Mean  Uncertainty
0.100000 0.550198 0.361084 0.518985 0.713578 0.000000  -0.365504     0.020339
0.250000 0.550380 0.365194 0.540448 0.708484 0.000000  -0.367191     0.029817
0.500000 0.549829 0.370954 0.581858 0.704243 0.000000  -0.374242     0.048511
0.750000 0.547564 0.377539 0.623681 0.701971 0.000000  -0.386627     0.068307
1.000000 0.543583 0.384153 0.665719 0.699280 0.000000  -0.404351     0.088555
1.500000 0.530761 0.391289 0.753622 0.686338 0.000000  -0.456377     0.130121

--- Optimized Kernel Parameters ---
Matern(length_scale=[1.5, 2.81, 3.04, 2.67, 3.05], nu=2.5) + WhiteKernel(noise_level=3.24e-10)


### Week 4 submission: 

Next point to query: [0.550377 , 0.365189 , 0.540408 , 0.708460 , 0.000000] beta = 0.25

* The last query was succesful, gained a good amount of "performance"
* Strategy is to keep optimising, keep an eye out for plateuing results 

In [11]:
# ---------------------------------------------------
# WEEK4: NEW DATA HERE
# ---------------------------------------------------

w4_new_input = 	[0.550377, 0.365189, 0.540408, 0.708460, 0.000000]
w4_new_output = -0.38370679276688424

w4_new_X = np.atleast_2d(w4_new_input)
w4_new_Y = np.atleast_1d(w4_new_output)

X_updated_w4 = np.vstack((X_updated_w3, w4_new_X))
Y_updated_w4 = np.append(Y_updated_w3, w4_new_Y)

X_updated_w4_scaled = scaler_x.fit_transform(X_updated_w4)
Y_updated_w4_scaled = scaler_y.fit_transform(Y_updated_w4.reshape(-1, 1))

df = pd.DataFrame(X_updated_w4, columns=['x1', 'x2', 'x3', 'x4', 'x5'])
df['target'] = Y_updated_w4

print(df.describe())

              x1         x2         x3         x4         x5     target
count  24.000000  24.000000  24.000000  24.000000  24.000000  24.000000
mean    0.541647   0.509382   0.492921   0.584571   0.366680  -1.339979
std     0.276170   0.289354   0.300624   0.283037   0.304477   0.556516
min     0.021735   0.061961   0.016523   0.045613   0.000000  -2.571170
25%     0.388851   0.290717   0.303186   0.380307   0.067588  -1.696397
50%     0.546946   0.396637   0.518617   0.679768   0.313578  -1.301682
75%     0.760851   0.796129   0.714228   0.747675   0.602252  -0.930201
max     0.957740   0.931871   0.978806   0.961873   0.892819  -0.369336


In [12]:
model.fit(X_updated_w4_scaled, Y_updated_w4_scaled)

betas = [0.05, 0.10 , 0.25, 0.50, 0.75, 1.0]
sweep_results = []

for b in betas:
    # Find the best starting point 
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    # Precision Climbing with L-BFGS-B
    bounds = [scaler_x.transform([[0,0,0,0,0]])[0], scaler_x.transform([[1,1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    # Translate predictions back to raw physical units
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)
    
    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'x5': best_x_raw[4],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })

df_sweep = pd.DataFrame(sweep_results)
print("--- FUNCTION 6: 5D CAKE RECIPE OPTIMIZATION ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)

--- FUNCTION 6: 5D CAKE RECIPE OPTIMIZATION ---
    Beta       x1       x2       x3       x4       x5  Pred Mean  Uncertainty
0.050000 0.555371 0.339017 0.433125 0.733252 0.000000  -0.360505     0.022757
0.100000 0.555396 0.339405 0.429016 0.731377 0.000000  -0.360625     0.024357
0.250000 0.554246 0.340564 0.415603 0.724491 0.000000  -0.361612     0.029903
0.500000 0.547674 0.343544 0.390065 0.710645 0.000000  -0.366362     0.042341
0.750000 0.536791 0.380154 0.353928 0.716868 0.000000  -0.377616     0.060083
1.000000 0.527593 0.423603 0.322959 0.735725 0.000000  -0.396940     0.082137

--- Optimized Kernel Parameters ---
Matern(length_scale=[1.53, 3.31, 3.2, 2.94, 3.25], nu=2.5) + WhiteKernel(noise_level=1e-10)


/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


### Week 5 submission:

Next point to query: [0.555371 0.339017 0.433125 0.733252 0.000000] beta = 0.05

* Hit a plateu, the performance is not improving - giving it another try with a lower beta
* If not improvement, go into exploration mode 

In [13]:
# ---------------------------------------------------
# WEEK5: NEW DATA HERE
# ---------------------------------------------------

w5_new_input = 	[0.555371, 0.339017, 0.433125, 0.733252, 0.000000]
w5_new_output = -0.4591096273863034

w5_new_X = np.atleast_2d(w5_new_input)
w5_new_Y = np.atleast_1d(w5_new_output)

X_updated_w5 = np.vstack((X_updated_w4, w5_new_X))
Y_updated_w5 = np.append(Y_updated_w4, w5_new_Y)

X_updated_w5_scaled = scaler_x.fit_transform(X_updated_w5)
Y_updated_w5_scaled = scaler_y.fit_transform(Y_updated_w5.reshape(-1, 1))

df = pd.DataFrame(X_updated_w5, columns=['x1', 'x2', 'x3', 'x4', 'x5'])
df['target'] = Y_updated_w5

print(df.describe())

              x1         x2         x3         x4         x5     target
count  25.000000  25.000000  25.000000  25.000000  25.000000  25.000000
mean    0.542196   0.502567   0.490529   0.590518   0.352013  -1.304744
std     0.270370   0.285304   0.294537   0.278668   0.306955   0.572576
min     0.021735   0.061961   0.016523   0.045613   0.000000  -2.571170
25%     0.432166   0.291470   0.341819   0.388225   0.056401  -1.694343
50%     0.550377   0.365189   0.496825   0.680515   0.298322  -1.294247
75%     0.757594   0.793678   0.708120   0.744821   0.598016  -0.913534
max     0.957740   0.931871   0.978806   0.961873   0.892819  -0.369336


In [14]:
model.fit(X_updated_w5_scaled, Y_updated_w5_scaled)

betas = [0.25, 0.50, 1.0, 1.5, 2.0, 2.5]
sweep_results = []

for b in betas:
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    # Precision Climbing with L-BFGS-B
    bounds = [scaler_x.transform([[0,0,0,0,0]])[0], scaler_x.transform([[1,1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)
    
    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'x5': best_x_raw[4],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })

df_sweep = pd.DataFrame(sweep_results)
print("--- FUNCTION 6: 5D CAKE RECIPE OPTIMIZATION ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("\n--- Optimized Kernel Parameters ---")
print(model.kernel_)

--- FUNCTION 6: 5D CAKE RECIPE OPTIMIZATION ---
    Beta       x1       x2       x3       x4       x5  Pred Mean  Uncertainty
0.250000 0.531423 0.372025 0.675935 0.735058 0.000000  -0.365790     0.075096
0.500000 0.528788 0.377169 0.709213 0.736403 0.000000  -0.370014     0.086217
1.000000 0.521949 0.387940 0.794047 0.735743 0.000000  -0.393297     0.116623
1.500000 0.513093 0.391307 0.913143 0.718944 0.000000  -0.448905     0.160555
2.000000 0.501868 0.388896 1.000000 0.692137 0.000000  -0.504852     0.194461
2.500000 0.445414 0.180291 1.000000 0.460796 0.000000  -0.667643     0.262586

--- Optimized Kernel Parameters ---
Matern(length_scale=[1.89, 2.66, 3.67, 3.12, 3.46], nu=2.5) + WhiteKernel(noise_level=0.00372)


### Week 6 submission

Next point to query: [1.500000 0.513093 0.391307 0.913143 0.718944 0.000000] beta = 1.50

* Changed strategy to exploration, the output was not improving for 2 rounds
* Choose a beta of 1.50 because the predicted mean was simialr to what was achieved with the much smaller beta (not strong reason)
* Try to understand and implement a different methodology, the approach of going for exploration in a 5D space is not promising

In [15]:
# ---------------------------------------------------
# WEEK6: NEW DATA HERE
# ---------------------------------------------------

w6_new_input = 	[0.513093, 0.391307, 0.913143, 0.718944, 0.000000]
w6_new_output = -0.5619970657433508

w6_new_X = np.atleast_2d(w6_new_input)
w6_new_Y = np.atleast_1d(w6_new_output)

X_updated_w6 = np.vstack((X_updated_w5, w6_new_X))
Y_updated_w6 = np.append(Y_updated_w5, w6_new_Y)

X_updated_w6_scaled = scaler_x.fit_transform(X_updated_w6)
Y_updated_w6_scaled = scaler_y.fit_transform(Y_updated_w6.reshape(-1, 1))

df = pd.DataFrame(X_updated_w6, columns=['x1', 'x2', 'x3', 'x4', 'x5'])
df['target'] = Y_updated_w6

# print(df.describe())
print(df.head(26))

          x1        x2        x3        x4        x5    target
0   0.728186  0.154693  0.732552  0.693997  0.056401 -0.714265
1   0.242384  0.844100  0.577809  0.679021  0.501953 -1.209955
2   0.729523  0.748106  0.679775  0.356552  0.671054 -1.672200
3   0.770620  0.114404  0.046780  0.648324  0.273549 -1.536058
4   0.618812  0.331802  0.187288  0.756238  0.328835 -0.829237
5   0.784958  0.910682  0.708120  0.959225  0.004911 -1.247049
6   0.145111  0.896685  0.896322  0.726272  0.236272 -1.233786
7   0.945069  0.288459  0.978806  0.961656  0.598016 -1.694343
8   0.125720  0.862725  0.028544  0.246605  0.751206 -2.571170
9   0.757594  0.355831  0.016523  0.434207  0.112433 -1.309116
10  0.536797  0.308781  0.411879  0.388225  0.522528 -1.144785
11  0.957740  0.235669  0.099146  0.156806  0.071317 -1.912677
12  0.629308  0.803484  0.811408  0.045613  0.110624 -1.622839
13  0.021735  0.428084  0.835939  0.489489  0.511082 -1.356682
14  0.439344  0.698924  0.426820  0.109476  0.877888 -2

In [16]:
threshold = np.percentile(Y_updated_w6, 50)
mask = Y_updated_w6 > threshold
X_exploit_scaled = X_updated_w6_scaled[mask]
Y_exploit_scaled = Y_updated_w6_scaled[mask]

active_indices = [1, 2, 3]
X_train_3d = X_exploit_scaled[:, active_indices]

y_train = Y_exploit_scaled.ravel() 

initial_length_scales = np.array([1.0] * 3)
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 100000.0), nu=2.5) + \
         WhiteKernel(noise_level=0.004, noise_level_bounds=(1e-5, 1e1))

model_3d = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=50,  
    random_state=42
)

model_3d.fit(X_train_3d, y_train)

def negative_mean(x_active, gp_model):
    mean = gp_model.predict(x_active.reshape(1, -1))
    return -mean[0]

best_idx = np.argmax(y_train)
x0_3d = X_train_3d[best_idx]

raw_lower_bound = [[0, 0, 0, 0, 0]]
raw_upper_bound = [[1, 1, 1, 1, 1]]
scaled_lower = scaler_x.transform(raw_lower_bound)[0]
scaled_upper = scaler_x.transform(raw_upper_bound)[0]

bounds_3d = []
for idx in active_indices:
    bounds_3d.append((scaled_lower[idx], scaled_upper[idx]))

res = minimize(
    fun=negative_mean,
    x0=x0_3d,
    args=(model_3d,),
    method='L-BFGS-B',
    bounds=bounds_3d
)

best_x_3d = res.x
predicted_max_mean_scaled = -res.fun

best_full_row_scaled = X_exploit_scaled[best_idx]
final_5d_scaled = np.zeros(5)

final_5d_scaled[0] = best_full_row_scaled[0]  # x1: Frozen (Ghost parameter)
final_5d_scaled[1] = best_x_3d[0]             # x2: Optimized
final_5d_scaled[2] = best_x_3d[1]             # x3: Optimized
final_5d_scaled[3] = best_x_3d[2]             # x4: Optimized
final_5d_scaled[4] = scaled_lower[4]          # x5: Frozen accurately at physical floor
 
final_5d_raw = scaler_x.inverse_transform(final_5d_scaled.reshape(1, -1))[0]
pred_mean_raw = scaler_y.inverse_transform([[predicted_max_mean_scaled]])[0][0]

print("\n--- NEXT PHYSICAL QUERY COORDINATES ---")
feature_names = ['x1', 'x2', 'x3', 'x4', 'x5']
for name, val in zip(feature_names, final_5d_raw):
    print(f"{val:.6f}")
print(f"\nPredicted Target Mean: {pred_mean_raw:.6f}")


--- NEXT PHYSICAL QUERY COORDINATES ---
0.543514
0.331248
0.639419
0.802612
0.000000

Predicted Target Mean: -0.271584


/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


### Week 7 submission

Next point to query [0.543514 0.331248 0.639419 0.802612 0.000000] 

* Changed the approach by training the model with just the top 50 percentile and froze x5 and turned the function into a 4D model. 

In [17]:
# ---------------------------------------------------
# WEEK7: NEW DATA HERE
# ---------------------------------------------------

w7_new_input = 	[0.543514, 0.331248, 0.639419, 0.802612, 0.000000]
w7_new_output = -0.35116605052151756

w7_new_X = np.atleast_2d(w7_new_input)
w7_new_Y = np.atleast_1d(w7_new_output)

X_updated_w7 = np.vstack((X_updated_w6, w7_new_X))
Y_updated_w7 = np.append(Y_updated_w6, w7_new_Y)

X_updated_w7_scaled = scaler_x.fit_transform(X_updated_w7)
Y_updated_w7_scaled = scaler_y.fit_transform(Y_updated_w7.reshape(-1, 1))

df = pd.DataFrame(X_updated_w7, columns=['x1', 'x2', 'x3', 'x4', 'x5'])
df['target'] = Y_updated_w7

print(df.describe())
# print(df.head(37))

              x1         x2         x3         x4         x5     target
count  27.000000  27.000000  27.000000  27.000000  27.000000  27.000000
mean    0.541167   0.492101   0.511696   0.603130   0.325938  -1.241917
std     0.259823   0.276818   0.295526   0.271812   0.309515   0.595581
min     0.021735   0.061961   0.016523   0.045613   0.000000  -2.571170
25%     0.435755   0.300126   0.376849   0.411216   0.018713  -1.683272
50%     0.543514   0.365189   0.540408   0.693997   0.273549  -1.247049
75%     0.743558   0.770892   0.720336   0.750530   0.595560  -0.771751
max     0.957740   0.931871   0.978806   0.961873   0.892819  -0.351166


In [18]:
threshold = np.percentile(Y_updated_w7, 50)
mask = Y_updated_w7 > threshold
X_exploit_scaled = X_updated_w7_scaled[mask]
Y_exploit_scaled = Y_updated_w7_scaled[mask]

active_indices = [1, 2, 3]
X_train_3d = X_exploit_scaled[:, active_indices]

y_train = Y_exploit_scaled.ravel() 

initial_length_scales = np.array([1.0] * 3)
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 100000.0), nu=2.5) + \
         WhiteKernel(noise_level=0.004, noise_level_bounds=(1e-5, 1e1))

model_3d = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=50,  
    random_state=42
)

model_3d.fit(X_train_3d, y_train)

def negative_mean(x_active, gp_model):
    mean = gp_model.predict(x_active.reshape(1, -1))
    return -mean[0]

best_idx = np.argmax(y_train)
x0_3d = X_train_3d[best_idx]

raw_lower_bound = [[0, 0, 0, 0, 0]]
raw_upper_bound = [[1, 1, 1, 1, 1]]
scaled_lower = scaler_x.transform(raw_lower_bound)[0]
scaled_upper = scaler_x.transform(raw_upper_bound)[0]

bounds_3d = []
for idx in active_indices:
    bounds_3d.append((scaled_lower[idx], scaled_upper[idx]))

res = minimize(
    fun=negative_mean,
    x0=x0_3d,
    args=(model_3d,),
    method='L-BFGS-B',
    bounds=bounds_3d
)

best_x_3d = res.x
predicted_max_mean_scaled = -res.fun

best_full_row_scaled = X_exploit_scaled[best_idx]
final_5d_scaled = np.zeros(5)

final_5d_scaled[0] = best_full_row_scaled[0]  # x1: Frozen (Ghost parameter)
final_5d_scaled[1] = best_x_3d[0]             # x2: Optimized
final_5d_scaled[2] = best_x_3d[1]             # x3: Optimized
final_5d_scaled[3] = best_x_3d[2]             # x4: Optimized
final_5d_scaled[4] = scaled_lower[4]          # x5: Frozen accurately at physical floor
 
final_5d_raw = scaler_x.inverse_transform(final_5d_scaled.reshape(1, -1))[0]
pred_mean_raw = scaler_y.inverse_transform([[predicted_max_mean_scaled]])[0][0]

print("\n--- NEXT PHYSICAL QUERY COORDINATES ---")
formatted_array = "[" + ", ".join([f"{val:.6f}" for val in final_5d_raw]) + "]"
print(formatted_array)
print(f"\nPredicted Target Mean: {pred_mean_raw:.6f}")


--- NEXT PHYSICAL QUERY COORDINATES ---
[0.543514, 0.330282, 0.607129, 0.755888, 0.000000]

Predicted Target Mean: -0.328516


### Week 8 submission

Next point to query [0.543514, 0.330282, 0.607129, 0.755888, 0.000000] 

* New best point! 

In [19]:
# ---------------------------------------------------
# WEEK8: NEW DATA HERE
# ---------------------------------------------------

w8_new_input = 	[0.543514, 0.330282, 0.607129, 0.755888, 0.000000]
w8_new_output = -0.28688801793206475

w8_new_X = np.atleast_2d(w8_new_input)
w8_new_Y = np.atleast_1d(w8_new_output)

X_updated_w8 = np.vstack((X_updated_w7, w8_new_X))
Y_updated_w8 = np.append(Y_updated_w7, w8_new_Y)

X_updated_w8_scaled = scaler_x.fit_transform(X_updated_w8)
Y_updated_w8_scaled = scaler_y.fit_transform(Y_updated_w8.reshape(-1, 1))

df = pd.DataFrame(X_updated_w8, columns=['x1', 'x2', 'x3', 'x4', 'x5'])
df['target'] = Y_updated_w8

print(df.describe())

              x1         x2         x3         x4         x5     target
count  28.000000  28.000000  28.000000  28.000000  28.000000  28.000000
mean    0.541251   0.486322   0.515104   0.608586   0.314297  -1.207809
std     0.254967   0.273359   0.290562   0.268289   0.309912   0.611681
min     0.021735   0.061961   0.016523   0.045613   0.000000  -2.571170
25%     0.437550   0.304453   0.394364   0.422712   0.008972  -1.677736
50%     0.543514   0.360510   0.540604   0.699498   0.254911  -1.240418
75%     0.736541   0.759499   0.714228   0.755976   0.594331  -0.681978
max     0.957740   0.931871   0.978806   0.961873   0.892819  -0.286888


In [20]:
threshold = np.percentile(Y_updated_w8, 50)
mask = Y_updated_w8 > threshold
X_exploit_scaled = X_updated_w8_scaled[mask]
Y_exploit_scaled = Y_updated_w8_scaled[mask]

active_indices = [1, 2, 3]
X_train_3d = X_exploit_scaled[:, active_indices]

y_train = Y_exploit_scaled.ravel() 

initial_length_scales = np.array([1.0] * 3)
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 100000.0), nu=2.5) + \
         WhiteKernel(noise_level=0.004, noise_level_bounds=(1e-5, 1e1))

model_3d = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=50,  
    random_state=42
)

model_3d.fit(X_train_3d, y_train)

def negative_mean(x_active, gp_model):
    mean = gp_model.predict(x_active.reshape(1, -1))
    return -mean[0]

best_idx = np.argmax(y_train)
x0_3d = X_train_3d[best_idx]

raw_lower_bound = [[0, 0, 0, 0, 0]]
raw_upper_bound = [[1, 1, 1, 1, 1]]
scaled_lower = scaler_x.transform(raw_lower_bound)[0]
scaled_upper = scaler_x.transform(raw_upper_bound)[0]

bounds_3d = []
for idx in active_indices:
    bounds_3d.append((scaled_lower[idx], scaled_upper[idx]))

res = minimize(
    fun=negative_mean,
    x0=x0_3d,
    args=(model_3d,),
    method='L-BFGS-B',
    bounds=bounds_3d
)

best_x_3d = res.x
predicted_max_mean_scaled = -res.fun

best_full_row_scaled = X_exploit_scaled[best_idx]
final_5d_scaled = np.zeros(5)

final_5d_scaled[0] = best_full_row_scaled[0]  # x1: Frozen (Ghost parameter)
final_5d_scaled[1] = best_x_3d[0]             # x2: Optimized
final_5d_scaled[2] = best_x_3d[1]             # x3: Optimized
final_5d_scaled[3] = best_x_3d[2]             # x4: Optimized
final_5d_scaled[4] = scaled_lower[4]          # x5: Frozen accurately at physical floor
 
final_5d_raw = scaler_x.inverse_transform(final_5d_scaled.reshape(1, -1))[0]
pred_mean_raw = scaler_y.inverse_transform([[predicted_max_mean_scaled]])[0][0]

print("\n--- NEXT PHYSICAL QUERY COORDINATES ---")
formatted_array = "[" + ", ".join([f"{val:.6f}" for val in final_5d_raw]) + "]"
print(formatted_array)
print(f"\nPredicted Target Mean: {pred_mean_raw:.6f}")


--- NEXT PHYSICAL QUERY COORDINATES ---
[0.543514, 0.313723, 0.633590, 0.738290, 0.000000]

Predicted Target Mean: -0.272171


/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


### Week 9 submission

Next point to query [0.543514, 0.313723, 0.633590, 0.738290, 0.000000]

* New max point
* Model is working

In [21]:
# ---------------------------------------------------
# WEEK9: NEW DATA HERE
# ---------------------------------------------------

w9_new_input = [0.543514, 0.313723, 0.633590, 0.738290, 0.000000]
w9_new_output = -0.35614621674381997

w9_new_X = np.atleast_2d(w9_new_input)
w9_new_Y = np.atleast_1d(w9_new_output)

X_updated_w9 = np.vstack((X_updated_w8, w9_new_X))
Y_updated_w9 = np.append(Y_updated_w8, w9_new_Y)

X_updated_w9_scaled = scaler_x.fit_transform(X_updated_w9)
Y_updated_w9_scaled = scaler_y.fit_transform(Y_updated_w9.reshape(-1, 1))

df = pd.DataFrame(X_updated_w9, columns=['x1', 'x2', 'x3', 'x4', 'x5'])
df['target'] = Y_updated_w9

print(df.describe())
# print(df.head(39))

              x1         x2         x3         x4         x5     target
count  29.000000  29.000000  29.000000  29.000000  29.000000  29.000000
mean    0.541329   0.480370   0.519190   0.613058   0.303460  -1.178441
std     0.250373   0.270340   0.286173   0.264553   0.309873   0.621130
min     0.021735   0.061961   0.016523   0.045613   0.000000  -2.571170
25%     0.439344   0.308781   0.411879   0.434207   0.004911  -1.672200
50%     0.543514   0.355831   0.540800   0.705000   0.236272  -1.233786
75%     0.729523   0.748106   0.708120   0.755888   0.593103  -0.585119
max     0.957740   0.931871   0.978806   0.961873   0.892819  -0.286888


In [22]:
threshold = np.percentile(Y_updated_w9, 50)
mask = Y_updated_w9 > threshold
X_exploit_scaled = X_updated_w9_scaled[mask]
Y_exploit_scaled = Y_updated_w9_scaled[mask]

active_indices = [1, 2, 3]
X_train_3d = X_exploit_scaled[:, active_indices]

y_train = Y_exploit_scaled.ravel() 

initial_length_scales = np.array([1.0] * 3)
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 100000.0), nu=2.5) + \
         WhiteKernel(noise_level=0.004, noise_level_bounds=(1e-5, 1e1))

model_3d = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=50,  
    random_state=42
)

model_3d.fit(X_train_3d, y_train)

def negative_mean(x_active, gp_model):
    mean = gp_model.predict(x_active.reshape(1, -1))
    return -mean[0]

best_idx = np.argmax(y_train)
x0_3d = X_train_3d[best_idx]

raw_lower_bound = [[0, 0, 0, 0, 0]]
raw_upper_bound = [[1, 1, 1, 1, 1]]
scaled_lower = scaler_x.transform(raw_lower_bound)[0]
scaled_upper = scaler_x.transform(raw_upper_bound)[0]

bounds_3d = []
for idx in active_indices:
    bounds_3d.append((scaled_lower[idx], scaled_upper[idx]))

res = minimize(
    fun=negative_mean,
    x0=x0_3d,
    args=(model_3d,),
    method='L-BFGS-B',
    bounds=bounds_3d
)

best_x_3d = res.x
predicted_max_mean_scaled = -res.fun

best_full_row_scaled = X_exploit_scaled[best_idx]
final_5d_scaled = np.zeros(5)

final_5d_scaled[0] = best_full_row_scaled[0]  # x1: Frozen (Ghost parameter)
final_5d_scaled[1] = best_x_3d[0]             # x2: Optimized
final_5d_scaled[2] = best_x_3d[1]             # x3: Optimized
final_5d_scaled[3] = best_x_3d[2]             # x4: Optimized
final_5d_scaled[4] = scaled_lower[4]          # x5: Frozen accurately at physical floor
 
final_5d_raw = scaler_x.inverse_transform(final_5d_scaled.reshape(1, -1))[0]
pred_mean_raw = scaler_y.inverse_transform([[predicted_max_mean_scaled]])[0][0]

print("\n--- NEXT PHYSICAL QUERY COORDINATES ---")
formatted_array = "[" + ", ".join([f"{val:.6f}" for val in final_5d_raw]) + "]"
print(formatted_array)
print(f"\nPredicted Target Mean: {pred_mean_raw:.6f}")


--- NEXT PHYSICAL QUERY COORDINATES ---
[0.543514, 0.334771, 0.610884, 0.762110, 0.000000]

Predicted Target Mean: -0.323016


### Week 10 submission

Next point to query [0.543514, 0.334771, 0.610884, 0.762110, 0.000000]

* Didn't improve, try again
* Model is working